In [ ]:
#cadprodofc

import tkinter as tk
from tkinter import ttk, messagebox
import firebase_admin
from firebase_admin import credentials, db
from datetime import datetime


class CadastroProdutosApp:
    def __init__(self):
        self.janela = tk.Tk()
        self.janela.title("Cadastro de Produtos - Estoque Pro")
        
        self.configurar_janela()
        self.db_ref = self.inicializar_firebase()
        
        if self.db_ref:
            self.criar_interface()
            self.janela.mainloop()
        else:
            self.janela.destroy()
    
    def configurar_janela(self):
        largura = 650
        altura = 700
        
        self.janela.geometry(f"{largura}x{altura}")
        self.janela.configure(bg="#C2C2C2")
        
        self.janela.update_idletasks()
        largura_tela = self.janela.winfo_screenwidth()
        altura_tela = self.janela.winfo_screenheight()
        posx = (largura_tela // 2) - (largura // 2)
        posy = (altura_tela // 2) - (altura // 2)
        self.janela.geometry(f"{largura}x{altura}+{posx}+{posy}")
        
        self.janela.grid_rowconfigure(1, weight=1)
        self.janela.grid_columnconfigure(0, weight=1)
    
    def inicializar_firebase(self):
        try:
            if not firebase_admin._apps:
                cred = credentials.Certificate("bancochave.json")
                firebase_admin.initialize_app(cred, {
                    'databaseURL': "https://bancoback-3c307-default-rtdb.firebaseio.com/"
                })

            print(" Firebase conectado com sucesso!")
            return db.reference("produtos")
            
        except FileNotFoundError:
            messagebox.showerror("Erro", "Arquivo 'bancochave.json' não encontrado!")
            return None
            
        except Exception as e:
            messagebox.showerror("Erro Firebase", f"Falha na conexão:\n{str(e)}")
            return None
    
    def criar_interface(self):

        header = tk.Frame(self.janela, bg="#236591", height=150)
        header.grid(row=0, column=0, sticky="ew")
        header.grid_propagate(False)
        
        titulo_frame = tk.Frame(header, bg="#236591")
        titulo_frame.pack(side="left", fill="y", padx=20)
        
        titulo = tk.Label(titulo_frame, text="Estoque Pro",
                          bg="#236591", fg="white", font=("Arial", 22, "bold"))
        titulo.pack(pady=(15, 5))
        
        subtitulo = tk.Label(titulo_frame, text="Controle total, resultado real",
                             fg="white", bg="#236591", font=("Arial", 12))
        subtitulo.pack(pady=(0, 15))
        
        main_frame = tk.Frame(self.janela, bg="#C2C2C2")
        main_frame.grid(row=1, column=0, padx=20, pady=(10, 20))
        
        titulo_form = tk.Label(main_frame, text="Cadastro de Produtos",
                               bg="#C2C2C2", fg="#236591", font=("Arial", 18, "bold"))
        titulo_form.pack(pady=(0, 15))
        
        form_frame = tk.Frame(main_frame, bg="white", relief="ridge", bd=2)
        form_frame.pack(fill="both", expand=True, padx=10, pady=10)
        
        style = ttk.Style()
        style.configure("TLabel", font=("Arial", 11), background="white")
        style.configure("TEntry", font=("Arial", 11), padding=5)
        
        labels = [
            "Código do Produto:",
            "Nome do Produto:",
            "Descrição do Produto:",
            "Unidade (un, kg, pç):",
            "Quantidade:",
            "Quantidade Mínima:",
            "Fornecedor:",
            "Importância:"
        ]
        
        self.entries = {}
        
        for i, texto in enumerate(labels):
            lbl = ttk.Label(form_frame, text=texto)
            lbl.grid(row=i, column=0, sticky="e", pady=8, padx=(20, 10))
            
            if texto == "Importância:":
                combo = ttk.Combobox(form_frame, values=["Baixa", "Média", "Alta"],
                                     state="readonly", width=33)
                combo.set("Selecione...")
                combo.grid(row=i, column=1, pady=8, padx=(0, 20))
                self.entries[texto] = combo
            else:
                ent = ttk.Entry(form_frame, width=35)
                ent.grid(row=i, column=1, pady=8, padx=(0, 20))
                self.entries[texto] = ent
        
        botao_frame = tk.Frame(form_frame, bg="white")
        botao_frame.grid(row=len(labels), column=0, columnspan=2, pady=(20, 10))
        
        btn_salvar = ttk.Button(botao_frame, text="Salvar",
                                command=self.salvar_produto, width=12)
        btn_salvar.grid(row=0, column=0, padx=10)
        
        btn_novo = ttk.Button(botao_frame, text="Novo",
                              command=self.limpar_campos, width=12)
        btn_novo.grid(row=0, column=1, padx=10)
        
        btn_sair = ttk.Button(botao_frame, text="Sair",
                              command=self.janela.destroy, width=12)
        btn_sair.grid(row=0, column=2, padx=10)
    
    def salvar_produto(self):

        codigo = self.entries["Código do Produto:"].get().strip()
        nome = self.entries["Nome do Produto:"].get().strip()
        descricao = self.entries["Descrição do Produto:"].get().strip()
        unidade = self.entries["Unidade (un, kg, pç):"].get().strip()
        quantidade = self.entries["Quantidade:"].get().strip()
        quantidade_minima = self.entries["Quantidade Mínima:"].get().strip()
        fornecedor = self.entries["Fornecedor:"].get().strip()
        importancia = self.entries["Importância:"].get().strip()
        
        if not codigo or not nome:
            messagebox.showwarning("Aviso", "Preencha Código e Nome!")
            return
        
        try:
            qtd = int(quantidade) if quantidade else 0
            qtd_min = int(quantidade_minima) if quantidade_minima else 0
            
            unidade_valida = unidade.lower() if unidade else 'un'
            if unidade_valida not in ['un', 'kg', 'pç', 'pc']:
                unidade_valida = 'un'
            
            dados_produto = {
                "codigo": codigo,
                "nome": nome,
                "descricao": descricao,
                "unidade": unidade_valida,
                "quantidade": qtd,
                "quantidade_minima": qtd_min,
                "fornecedor": fornecedor,
                "importancia": importancia,
                "data_cadastro": {'.sv': 'timestamp'},
                "ativo": True,
                "ultima_atualizacao": {'.sv': 'timestamp'}
            }
        
            produtos = self.db_ref.get()
            codigo_existe = False
            
            if produtos:
                for pid, pdata in produtos.items():
                    if pdata.get('codigo') == codigo:
                        codigo_existe = True
                        produto_id = pid
                        break
            
            if codigo_existe:
                self.db_ref.child(produto_id).update(dados_produto)
                messagebox.showinfo("Atualizado", "Produto atualizado com sucesso!")
            else:
                novo = self.db_ref.push(dados_produto)
                messagebox.showinfo("Cadastrado", f"Produto salvo!\nID: {novo.key}")
            
            self.limpar_campos()
            
        except ValueError:
            messagebox.showerror("Erro", "Quantidade deve ser número inteiro.")
    
    def limpar_campos(self):
        for entry in self.entries.values():
            entry.delete(0, tk.END)
        self.entries["Importância:"].set("Selecione...")
        self.entries["Código do Produto:"].focus_set()


if __name__ == "__main__":
    app = CadastroProdutosApp()


 Firebase conectado com sucesso!


In [7]:
#Quantidade - Estoque

import tkinter as tk
from tkinter import ttk, messagebox
import firebase_admin
from firebase_admin import credentials, db
from datetime import datetime

def inicializar_firebase():
    try:
        if not firebase_admin._apps:
            cred = credentials.Certificate("bancochave.json")
            firebase_admin.initialize_app(cred, {
                'databaseURL': "https://bancodedadosprojeto-b4cec-default-rtdb.firebaseio.com/"
            })
        return db.reference("produtos"), db.reference("movimentacoes")
    except Exception as e:
        messagebox.showerror("Erro ao conectar Firebase", str(e))
        return None, None

janela = tk.Tk()
janela.title("Produtos Cadastrados")
janela.geometry("750x500")
janela.configure(bg="#C2C2C2")
janela.grid_rowconfigure(1, weight=1)
janela.grid_columnconfigure(0, weight=1)

header = tk.Frame(janela, bg="#236591", height=120)
header.grid(row=0, column=0, sticky="ew")
header.grid_propagate(False)
header.grid_columnconfigure(0, weight=1)

titulo = tk.Label(header, text="Estoque Pro", bg="#236591", fg="white", font=("Arial", 22, "bold"))
titulo.grid(row=0, column=0, pady=(15, 0))

subtitulo = tk.Label(header, text="Controle total, resultado real", bg="#236591", fg="white", font=("Arial", 12))
subtitulo.grid(row=1, column=0, pady=(0, 15))

colunas = ("Produto", "Quantidade")
tabela = ttk.Treeview(janela, columns=colunas, show="headings", height=10)
for col in colunas:
    tabela.heading(col, text=col)
    tabela.column(col, width=200)
tabela.grid(row=1, column=0, sticky="nsew", padx=10, pady=10)

scrollbar = ttk.Scrollbar(janela, orient="vertical", command=tabela.yview)
tabela.configure(yscroll=scrollbar.set)
scrollbar.grid(row=1, column=1, sticky="ns")

def carregar_produtos():
    tabela.delete(*tabela.get_children())
    ref_produtos, ref_movimentacoes = inicializar_firebase()
    if ref_produtos is None:
        return
    dados = ref_produtos.get()
    if dados:
        for id_produto, item in dados.items():
            produto_nome = item.get("nome", "-") 
            quantidade = item.get("quantidade", 0) 
            tabela.insert("", "end", iid=id_produto, values=(produto_nome, quantidade))
    else:
        messagebox.showinfo("Aviso", "Nenhum produto cadastrado.")

def registrar_historico(nome, tipo, qtd):
    try:
        ref_produtos, ref_movimentacoes = inicializar_firebase()
        historico_ref = ref_movimentacoes  
        hora_atual = datetime.now().strftime("%d/%m/%Y %H:%M:%S")
        historico_ref.push({
            "produto": nome,
            "tipo": tipo,
            "quantidade": qtd,
            "data_hora": hora_atual
        })
    except Exception as e:
        messagebox.showerror("Erro", f"Erro ao registrar histórico: {e}")

def alterar_quantidade(tipo):
    selecionado = tabela.focus()
    if not selecionado:
        messagebox.showwarning("Aviso", "Selecione um produto.")
        return
    ref_produtos, ref_movimentacoes = inicializar_firebase()
    if ref_produtos is None:
        return
    produto_ref = ref_produtos.child(selecionado)
    produto = produto_ref.get() 

    if produto is None:
        messagebox.showerror("Erro", "Produto não encontrado no banco.")
        return

    nome = produto.get("nome", "-")  
    qtd_atual = produto.get("quantidade", 0)  
    qtd = qtd_input.get()

    if not qtd.isdigit() or int(qtd) <= 0:
        messagebox.showwarning("Aviso", "Digite uma quantidade válida.")
        return

    qtd = int(qtd)
    if tipo == "remover":
        if qtd > qtd_atual:
            messagebox.showerror("Erro", "A quantidade a remover é maior que o estoque!")
            return
        nova_qtd = qtd_atual - qtd
    else:
        nova_qtd = qtd_atual + qtd

    produto_ref.update({"quantidade": nova_qtd})

    registrar_historico(nome, "Entrada" if tipo == "adicionar" else "Saída", qtd)

    carregar_produtos()
    qtd_input.delete(0, tk.END)

def abrir_historico():
    hist_janela = tk.Toplevel()
    hist_janela.title("Histórico de Movimentações")
    hist_janela.geometry("750x400")
    hist_janela.configure(bg="#D9D9D9")
    colunas = ("Produto", "Tipo", "Quantidade", "Data/Hora")
    tabela_hist = ttk.Treeview(hist_janela, columns=colunas, show="headings", height=15)
    for col in colunas:
        tabela_hist.heading(col, text=col)
        tabela_hist.column(col, width=150)
    tabela_hist.pack(fill="both", expand=True, padx=10, pady=10)

    scrollbar = ttk.Scrollbar(hist_janela, orient="vertical", command=tabela_hist.yview)
    tabela_hist.configure(yscroll=scrollbar.set)
    scrollbar.pack(side="right", fill="y")

    try:
        dados = db.reference("movimentacoes").get()
        if dados:
            for item in dados.values():
                tabela_hist.insert("", "end", values=(item.get("produto", "-"), item.get("tipo", "-"),
                                                      item.get("quantidade", "-"), item.get("data_hora", "-")))
        else:
            messagebox.showinfo("Histórico", "Nenhuma movimentação registrada.")
    except Exception as e:
        messagebox.showerror("Erro", f"Erro ao carregar histórico: {e}")

controle_frame = tk.Frame(janela, bg="#C2C2C2")
controle_frame.grid(row=2, column=0, pady=10)

tk.Label(controle_frame, text="Quantidade:", bg="#C2C2C2").grid(row=0, column=0, padx=5)
qtd_input = tk.Entry(controle_frame, width=10)
qtd_input.grid(row=0, column=1, padx=5)

btn_remover = tk.Button(controle_frame, text="Remover", bg="#D9534F", fg="white", command=lambda: alterar_quantidade("remover"))
btn_remover.grid(row=0, column=2, padx=10)

btn_adicionar = tk.Button(controle_frame, text="Adicionar", bg="#5CB85C", fg="white", command=lambda: alterar_quantidade("adicionar"))
btn_adicionar.grid(row=0, column=3, padx=10)

btn_atualizar = tk.Button(controle_frame, text="Atualizar Tabela", command=carregar_produtos)
btn_atualizar.grid(row=0, column=4, padx=10)

btn_historico = tk.Button(controle_frame, text="Histórico", bg="#0275D8", fg="white", command=abrir_historico)
btn_historico.grid(row=0, column=5, padx=10)

carregar_produtos()
janela.mainloop()